In [ ]:
# uncomment if you wish to run this notebook on google colab with dataset on google drive

# from google.colab import drive
# drive.mount('/content/drive')

# Global Variables

In [ ]:
dataset_path = "path/to/dataset/folder"

In [ ]:
allSubjects = [f"Subject_{i+1:02d}" for i in range(20)]

In [ ]:
N_EMG_CHANNEL = 8
N_KINEMATIC_CHANNEL = 15
EMG_S_FREQ = 500

# Utils

## General imports and setup

In [ ]:
!pip install pyriemann
!pip install mne

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pyriemann.tangentspace import TangentSpace
from pyriemann.estimation import Covariances
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
import mne
from keras.src.callbacks import EarlyStopping
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras import backend as K
import gc
from copy import deepcopy
from scipy.ndimage import gaussian_filter1d
import seaborn as sn

In [ ]:
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.optimizers import Adam
import tensorflow as tf

In [ ]:
from tensorflow.keras import mixed_precision
mixed_precision.set_global_policy("mixed_float16")

In [ ]:
!nvidia-smi
tf.config.list_physical_devices('GPU')

## Data loading

In [ ]:
def loadData(sName):
  data = mne.io.read_raw_fif(f"{dataset_path}/{sName}.fif", preload=True)

  emg = data.get_data(picks=[f"EMG {i+1}" for i in range(N_EMG_CHANNEL)]).T
  label = data.get_data(picks=[f"Angle {i+1}" for i in range(N_KINEMATIC_CHANNEL)]).T

  emg = mne.filter.filter_data(emg.T, EMG_S_FREQ, 15, 150, verbose="CRITICAL")
  emg = mne.filter.notch_filter(emg, EMG_S_FREQ, 50, notch_widths=5, verbose="CRITICAL")
  emg = mne.filter.notch_filter(emg, EMG_S_FREQ, 100, notch_widths=5, verbose="CRITICAL")
  emg = mne.filter.notch_filter(emg, EMG_S_FREQ, 150, notch_widths=5, verbose="CRITICAL").T

  return emg, label

## Synchronization

In [ ]:
def sync(emg, label):
  sfreq = EMG_S_FREQ
  ch_names = [f"emg{i+1}" for i in range(N_EMG_CHANNEL)]
  info = mne.create_info(ch_names=ch_names, sfreq=sfreq, ch_types="emg")
  raw = mne.io.RawArray(deepcopy(emg).T, info)
  raw.filter(15, 50, picks=ch_names)

  raw_envelope = raw.apply_hilbert(envelope=True, picks=ch_names)
  envelope = raw_envelope.get_data()

  # decomposition into move and hold commands
  S = envelope
  M = np.zeros(S.shape)
  H = np.zeros(S.shape)

  for i in range(1, S.shape[1]):
      M[:, i] = S[:, i] - H[:, i-1]
      H[:, i] = H[:, i-1] + M[:, i]/sfreq

  l = np.apply_along_axis(gaussian_filter1d, 0, label, sigma=20)
  speed = np.gradient(l)[0]
  speed = np.apply_along_axis(gaussian_filter1d, 0, speed, sigma=20)

  M2 = np.apply_along_axis(gaussian_filter1d, 0, M.T, sigma=20).T

  CUT = 4
  shiftPerSegment = []

  for i in range(CUT):
    idx = np.arange(len(speed)*i//CUT, len(speed)*(i+1)//CUT)

    M2_ = M2[:, idx]
    speed_ = speed[idx]

    shift = 0
    cor = []
    l = len(speed_)
    for shift in range(-200, 501, 5):
      if shift >=0:
        a = np.max(M2_[:, :l-shift]**2, axis=0)**0.5
        b = np.max(speed_[shift:]**2, axis=1)**0.5
      else:
        a = np.max(M2_[:, -shift:]**2, axis=0)**0.5
        b = np.max(speed_[:l+shift]**2, axis=1)**0.5

      cor.append((shift, np.corrcoef(a, b)[0, 1]))
    cor = np.array(cor)

    m = cor[np.argmax(cor[:, 1]), 0]
    shiftPerSegment.append(m)

  label_sync = np.concatenate([label[np.array(np.clip(np.arange(len(speed)*i//CUT, len(speed)*(i+1)//CUT) + shiftPerSegment[i], 0, len(label)-1), dtype=int)] for i in range(CUT)])
  return label_sync

## Sample Extraction

In [ ]:
def extractSamples_10fps(emg, label, shift=0, size=150, labelSequence=False):
    step = 50
    ts = np.arange(2000, len(emg)-2000, step)

    x_emg = []
    y = []
    for t in ts:
        x_emg.append(emg[t - size+1:t+1].transpose())
        if labelSequence:
            y.append(label[t - size+1:t+1].transpose())
        else:
            y.append(label[t+shift])
    x_emg = np.array(x_emg)
    y = np.array(y)

    return x_emg, y

In [ ]:
def createSequences(x, y, sequenceLength):
    sequence = []
    Y = []
    for i in range(sequenceLength, len(x)-1):
        sequence.append(x[i-sequenceLength+1:i+1])
        Y.append(y[i])
    sequence = np.array(sequence)
    y = np.array(Y)
    return sequence, y

# Feature extraction

In [ ]:
def rawEMG(xTrain, xTest):
  return xTrain, xTest

In [ ]:
def envelopeEMG(xTrain, xTest):
  ch_names = [f"emg{i+1}" for i in range(N_EMG_CHANNEL)]
  info = mne.create_info(ch_names=ch_names, sfreq=EMG_S_FREQ, ch_types="emg")
  xTrain = mne.EpochsArray(xTrain, info)
  xTest = mne.EpochsArray(xTest, info)

  xTrain = xTrain.copy().apply_hilbert(envelope=True, picks=ch_names).get_data()
  xTest = xTest.copy().apply_hilbert(envelope=True, picks=ch_names).get_data()

  return xTrain, xTest

In [ ]:
def timeDomainFeatures(xTrain, xTest):
  def MAV(x):
    return np.mean(np.abs(x), axis=2)

  def RMS(x):
    return np.sqrt(np.mean(x**2, axis=2))

  def MAA(x):
    return np.max(np.abs(x), axis=2)

  def WL(x):
    return np.sum(np.abs(x[:, :, 1:] - x[:, :, :-1]), axis=2)

  def SSC(x):
    return np.sum(np.sign((x[:, :, 2:] - x[:, :, 1:-1])*(x[:, :, 1:-1] - x[:, :, :-2])), axis=2)

  def WA(x):
    return np.sum(np.sign(np.abs(x[:, :, 1:] - x[:, :, :-1]) - np.std(x, axis=2).reshape((len(x), len(x[0]), 1))), axis=2)

  def MFL(x):
      res = np.log(np.sqrt(np.sum((x[:, :, 1:] - x[:, :, :-1])**2, axis=2)))
      res[res < -1000] = -1000
      return res

  TDF = [MAV, RMS, MAA, WL, SSC, WA, MFL]
  tdf_train = np.concatenate([f(xTrain.astype(np.float32)) for f in TDF], axis=1)
  tdf_test  = np.concatenate([f(xTest.astype(np.float32))  for f in TDF], axis=1)
  scaler = StandardScaler().fit(tdf_train)
  tdf_train = scaler.transform(tdf_train)
  tdf_test  = scaler.transform(tdf_test)
  return tdf_train, tdf_test

In [ ]:
def covMatTangentSpace(xTrain, xTest):
  ds = 3
  cmtsExtractor = Pipeline([('cov', Covariances(estimator='oas')), ('ts', TangentSpace('riemann'))])
  cmtsExtractor.fit(xTrain[::ds].astype(np.float32))
  cmts_train = []
  l = 10000
  for j in range(len(xTrain)//l +1):
    cmts_train.extend(cmtsExtractor.transform(xTrain[j*l:(j+1)*l].astype(np.float32)))
    print(j, "/", len(xTrain)//l +1)
  cmts_train = np.array(cmts_train)
  print("cmts train", cmts_train.shape)
  cmts_test = cmtsExtractor.transform(xTest.astype(np.float32))
  return cmts_train, cmts_test

In [ ]:
def covMatTangentSpace_3FreqBands(xTrain, xTest):
  """
  needs to have extracted samples in multiple frequency bands
  """
  ds = 3
  xTrainBands = []
  xTestBands = []
  for i in range(3):
    cmtsExtractor = Pipeline([('cov', Covariances(estimator='oas')), ('ts', TangentSpace('riemann'))])
    cmtsExtractor.fit(xTrain[::ds, i].astype(np.float32))
    cmts_train = []
    l = 10000
    for j in range(len(xTrain)//l +1):
      cmts_train.extend(cmtsExtractor.transform(xTrain[j*l:(j+1)*l, i].astype(np.float32)))
      print(j, "/", len(xTrain)//l +1)
    cmts_train = np.array(cmts_train)
    print("cmts train", cmts_train.shape)
    cmts_test = cmtsExtractor.transform(xTest[:, i].astype(np.float32))
    xTrainBands.append(cmts_train)
    xTestBands.append(cmts_test)
  cmts_train = np.concatenate(xTrainBands, axis=1)
  cmts_test = np.concatenate(xTestBands, axis=1)
  return cmts_train, cmts_test

# Regression Algorithms

## MLP

In [ ]:
def createMLP(output_dim=15, input_shape=None):
    model = keras.Sequential(name="MLP_Model")
    model.add(layers.Input(shape=input_shape))

    model.add(layers.Dense(256, activation="tanh"))
    model.add(layers.Flatten())

    model.add(layers.Dense(256, activation="tanh"))
    model.add(layers.Dense(128, activation="tanh"))

    model.add(layers.Dense(64, activation="tanh"))
    model.add(layers.Dense(output_dim, activation="linear", dtype="float32"))

    optimizer = Adam(learning_rate=2e-4, clipnorm=1.0)
    optimizer = mixed_precision.LossScaleOptimizer(optimizer)
    model.compile(optimizer=optimizer, loss="huber", metrics=["mae"])
    return model


tf.config.optimizer.set_jit(True)

In [ ]:
def make_dataset(X, Y, batchSize=256):
    ds = tf.data.Dataset.from_tensor_slices((X, Y))
    return ds.shuffle(len(X)).batch(batchSize).prefetch(tf.data.AUTOTUNE)

In [ ]:
def mlp(xTrain, yTrain, xTest):
    batchSize = 256

    trainIdx = np.arange(len(xTrain))
    valSize = len(xTrain)//10 # use 10% of training data for validation to find convergence
    valIdx_start = np.random.randint(len(xTrain) - valSize)
    valIdx = trainIdx[valIdx_start:valIdx_start+valSize]
    trainIdx = np.delete(trainIdx, np.arange(valIdx_start, valIdx_start+valSize))
    np.random.shuffle(trainIdx)

    # Data subsets
    trainX, trainY = xTrain[trainIdx], yTrain[trainIdx]
    valX, valY = xTrain[valIdx], yTrain[valIdx]

    # Model
    reg = createMLP(output_dim=15, input_shape=(xTrain.shape[1], xTrain.shape[2]))

    # Callbacks
    early_stop = EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True)
    lr_schedule = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6)

    train_ds = make_dataset(trainX, trainY, batchSize)
    val_ds = make_dataset(valX, valY, batchSize)

    history = reg.fit(
        train_ds,
        validation_data=val_ds,
        epochs=200,
        callbacks=[early_stop, lr_schedule],
        verbose=0
    )
    yPred = reg.predict(xTest, batch_size=512)


    plt.rcParams['figure.figsize'] = [5, 5]
    plt.plot(history.history['loss'], label="loss")
    plt.plot(history.history['val_loss'], label="val loss")
    plt.show()

    # cleanup
    K.clear_session()
    del reg
    gc.collect()

    return yPred

## RNN

In [ ]:
def createRNN(output_dim=15, input_shape=None):
    model = keras.Sequential(name="GRU_Model")
    model.add(layers.Input(shape=input_shape))

    model.add(layers.Dense(256, activation="tanh"))

    model.add(layers.GRU(256, return_sequences=True, dropout=0.1, activation="tanh"))
    model.add(layers.GRU(128, return_sequences=False, dropout=0.1, activation="tanh"))

    model.add(layers.Dense(64, activation="tanh"))
    model.add(layers.Dense(output_dim, activation="linear", dtype="float32"))

    optimizer = Adam(learning_rate=2e-4, clipnorm=1.0)
    optimizer = mixed_precision.LossScaleOptimizer(optimizer)
    model.compile(optimizer=optimizer, loss="huber", metrics=["mae"])
    return model


tf.config.optimizer.set_jit(True)

In [ ]:
def make_dataset(X, Y, batchSize=256):
    ds = tf.data.Dataset.from_tensor_slices((X, Y))
    return ds.shuffle(len(X)).batch(batchSize).prefetch(tf.data.AUTOTUNE)

In [ ]:
def rnn(xTrain, yTrain, xTest):
    batchSize = 256

    trainIdx = np.arange(len(xTrain))
    valSize = len(xTrain)//10 # use 10% of training data for validation to find convergence
    valIdx_start = np.random.randint(len(xTrain) - valSize)
    valIdx = trainIdx[valIdx_start:valIdx_start+valSize]
    trainIdx = np.delete(trainIdx, np.arange(valIdx_start, valIdx_start+valSize))
    np.random.shuffle(trainIdx)

    # Data subsets
    trainX, trainY = xTrain[trainIdx], yTrain[trainIdx]
    valX, valY = xTrain[valIdx], yTrain[valIdx]

    # Model
    reg = createRNN(output_dim=trainY.shape[-1], input_shape=(xTrain.shape[1], xTrain.shape[2]))

    # Callbacks
    early_stop = EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True)
    lr_schedule = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6)

    train_ds = make_dataset(trainX, trainY, batchSize)
    val_ds = make_dataset(valX, valY, batchSize)

    history = reg.fit(
        train_ds,
        validation_data=val_ds,
        epochs=200,
        callbacks=[early_stop, lr_schedule],
        verbose=1
    )
    yPred = reg.predict(xTest, batch_size=512)


    plt.rcParams['figure.figsize'] = [5, 5]
    plt.plot(history.history['loss'], label="loss")
    plt.plot(history.history['val_loss'], label="val loss")
    plt.show()

    # cleanup
    K.clear_session()
    del reg
    gc.collect()

    return yPred

## Deep CRNN (CNN -> RNN)

In [ ]:
class EMG_net(tf.keras.Model):
    def __init__(self, input_channels, output_dim):
        super().__init__()
        self.output_dim = output_dim
        self.conv1 = layers.Conv1D(filters=32, kernel_size=3, padding='same', activation='relu')
        self.maxPool1 = layers.MaxPool1D(pool_size=2, strides=2)
        self.conv2 = layers.Conv1D(filters=32, kernel_size=3, padding='same', activation='relu')
        self.maxPool1 = layers.MaxPool1D(pool_size=2, strides=2)
        self.conv3 = layers.Conv1D(filters=32, kernel_size=3, padding='same', activation='relu')
        self.maxPool1 = layers.MaxPool1D(pool_size=2, strides=2)
        self.conv4 = layers.Conv1D(filters=32, kernel_size=3, padding='same', activation='relu')
        self.maxPool1 = layers.MaxPool1D(pool_size=2, strides=2)
        self.channel_collapse = layers.Conv1D(filters=32, kernel_size=3, padding='same', activation='relu')
        self.normalization = layers.BatchNormalization(axis=1)
        self.dense_time = layers.TimeDistributed(layers.Dense(128, activation='relu'))
        self.gru1 = layers.GRU(128, return_sequences=True, dropout=0.1, activation="tanh")
        self.gru2 = layers.GRU(64, return_sequences=False, dropout=0.1, activation="tanh")
        self.post_dense1 = layers.Dense(64, activation="relu")
        self.final_dense = layers.Dense(self.output_dim, activation="linear", dtype="float32")


    def encoder(self, X):
        l = tf.shape(X)[2]
        b = tf.shape(X)[0]

        X = tf.transpose(X, perm=[0, 2, 1]) # (batch size, 8, 600, 16)
        X = self.conv1(X)
        X = self.maxPool1(X)
        X = self.conv2(X)
        X = self.maxPool1(X)
        X = self.conv3(X)
        X = self.maxPool1(X)
        X = self.conv4(X)
        X = self.channel_collapse(X)
        X = tf.transpose(X, perm=[0, 2, 1])
        X = self.normalization(X)

        return X

    def decoder(self, X):
        b = tf.shape(X)[0]

        X = tf.signal.frame(X, frame_length=20, frame_step=6, axis=2)
        X = tf.transpose(X, perm=[0, 2, 1, 3])
        X = tf.reshape(X, (b, tf.shape(X)[1], 32 * 20))
        X = self.dense_time(X)

        X = self.gru1(X)
        X = self.gru2(X)

        X = self.post_dense1(X)
        X = self.final_dense(X)

        return X


    def build(self, input_shape):
        dummy = tf.zeros((1, input_shape[1], input_shape[2]))
        dummy = self.encoder(dummy)
        dummy = self.decoder(dummy)
        super().build(input_shape)

    def call(self, X, return_conv2=False):
        X = self.encoder(X)
        if return_conv2:
            return X
        X = self.decoder(X)
        return X

tf.config.optimizer.set_jit(True)

In [ ]:

def deep_crnn(trainIdx, testIdx):
    batchSize = 256

    # Select one validation subject randomly
    valIdx = np.random.choice(trainIdx, 1)[0]
    trainIdx = np.delete(trainIdx, np.where(trainIdx == valIdx))

    # Generator to stream samples from subjects
    def subject_generator(subject_indices):
        for idx in subject_indices:
            x = np.load(f"X_{allSubjects[idx]}.npy")
            y = np.load(f"Y_{allSubjects[idx]}.npy")
            for i in range(len(x)):
                yield x[i], y[i]

    # Determine sample shapes from the first subject
    sample_x = np.load(f"X_{allSubjects[trainIdx[0]]}.npy")
    sample_y = np.load(f"Y_{allSubjects[trainIdx[0]]}.npy")
    x_shape = sample_x.shape[1:]  # exclude batch dimension
    y_shape = sample_y.shape[1:]

    # Training dataset
    train_dataset = tf.data.Dataset.from_generator(
        lambda: subject_generator(trainIdx),
        output_signature=(
            tf.TensorSpec(shape=x_shape, dtype=tf.float32),
            tf.TensorSpec(shape=y_shape, dtype=tf.float32)
        )
    ).shuffle(buffer_size=10000).batch(batchSize).prefetch(tf.data.AUTOTUNE)

    # Validation dataset
    val_dataset = tf.data.Dataset.from_generator(
        lambda: subject_generator([valIdx]),
        output_signature=(
            tf.TensorSpec(shape=x_shape, dtype=tf.float32),
            tf.TensorSpec(shape=y_shape, dtype=tf.float32)
        )
    ).batch(512).prefetch(tf.data.AUTOTUNE)

    # Model
    reg = EMG_net(input_channels=N_EMG_CHANNEL, output_dim=N_KINEMATIC_CHANNEL)
    reg.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=2e-4),
        loss=tf.keras.losses.Huber(),
        metrics=[tf.keras.metrics.MeanSquaredError()],
    )

    # Train
    early_stop = tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=10,
        restore_best_weights=True
    )

    reg.fit(
        train_dataset,
        validation_data=val_dataset,
        epochs=300,
        callbacks=[early_stop],
        verbose=1
    )

    # Predict on test subject
    test_x = np.load(f"X_{allSubjects[testIdx]}.npy")
    yPred = reg.predict(test_x, batch_size=512)

    # Cleanup
    K.clear_session()
    del reg
    gc.collect()

    return yPred

## vemg2pose

In [ ]:
import collections
from collections.abc import Sequence

from typing import Literal

import torch

from torch import nn

class Conv1dBlock(nn.Module):
    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        kernel_size: int,
        stride: int,
        norm_type: Literal["layer", "batch", "none"] = "layer",
        dropout: float = 0.0,
    ):
        """A 1D convolution with padding so the input and output lengths match."""

        super().__init__()

        self.norm_type = norm_type
        self.kernel_size = kernel_size
        self.stride = stride

        layers = {}
        layers["conv1d"] = nn.Conv1d(
            in_channels,
            out_channels,
            kernel_size=kernel_size,
            stride=stride,
            padding=0,
        )

        """if norm_type == "batch":
            layers["norm"] = BatchNorm1d(out_channels)""" # meta use "layer" normalization, not batch

        layers["relu"] = nn.ReLU(inplace=True)
        layers["dropout"] = nn.Dropout(dropout)

        self.conv = nn.Sequential(
            *[layers[key] for key in layers if layers[key] is not None]
        )

        if norm_type == "layer":
            self.norm = nn.LayerNorm(normalized_shape=out_channels)

    def forward(self, x):
        x = self.conv(x)
        if self.norm_type == "layer":
            x = self.norm(x.swapaxes(-1, -2)).swapaxes(-1, -2)
        return x

class TDSConv2dBlock(nn.Module):
    """A 2D temporal convolution block as per "Sequence-to-Sequence Speech
    Recognition with Time-Depth Separable Convolutions, Hannun et al"
    (https://arxiv.org/abs/1904.02619).

    Args:
        channels (int): Number of input and output channels. For an input of
            shape (T, N, num_features), the invariant we want is
            channels * width = num_features.
        width (int): Input width. For an input of shape (T, N, num_features),
            the invariant we want is channels * width = num_features.
        kernel_width (int): The kernel size of the temporal convolution.
    """

    def __init__(self, channels: int, width: int, kernel_width: int) -> None:
        super().__init__()

        assert kernel_width % 2, "kernel_width must be odd."
        self.conv2d = nn.Conv2d(
            in_channels=channels,
            out_channels=channels,
            kernel_size=(1, kernel_width),
            dilation=(1, 1),
            stride=(1, 1),
            padding=(0, 0),
            groups=1,
            bias=True,
        )
        self.relu = nn.ReLU(inplace=True)
        self.layer_norm = nn.LayerNorm(channels * width)

        self.channels = channels
        self.width = width

    def forward(self, inputs: torch.Tensor) -> torch.Tensor:

        B, C, T = inputs.shape  # BCT

        # BCT -> BcwT
        x = inputs.reshape(B, self.channels, self.width, T)
        x = self.conv2d(x)
        x = self.relu(x)
        x = x.reshape(B, C, -1)  # BcwT -> BCT

        # Skip connection after downsampling
        T_out = x.shape[-1]
        x = x + inputs[..., -T_out:]

        # Layer norm over C
        x = self.layer_norm(x.swapaxes(-1, -2)).swapaxes(-1, -2)

        return x


class TDSFullyConnectedBlock(nn.Module):
    """A fully connected block as per "Sequence-to-Sequence Speech
    Recognition with Time-Depth Separable Convolutions, Hannun et al"
    (https://arxiv.org/abs/1904.02619).

    Args:
        num_features (int): ``num_features`` for an input of shape
            (T, N, num_features).
    """

    def __init__(self, num_features: int) -> None:
        super().__init__()

        self.fc_block = nn.Sequential(
            nn.Linear(num_features, num_features),
            nn.ReLU(inplace=True),
            nn.Linear(num_features, num_features),
        )
        self.layer_norm = nn.LayerNorm(num_features)

    def forward(self, inputs: torch.Tensor) -> torch.Tensor:

        x = inputs
        x = x.swapaxes(-1, -2)  # BCT -> BTC
        x = self.fc_block(x)
        x = x.swapaxes(-1, -2)  # BTC -> BCT
        x += inputs

        # Layer norm over C
        x = self.layer_norm(x.swapaxes(-1, -2)).swapaxes(-1, -2)

        return x


class TDSConvEncoder(nn.Module):
    """A time depth-separable convolutional encoder composing a sequence
    of `TDSConv2dBlock` and `TDSFullyConnectedBlock` as per
    "Sequence-to-Sequence Speech Recognition with Time-Depth Separable
    Convolutions, Hannun et al" (https://arxiv.org/abs/1904.02619).

    Args:
        num_features (int): ``num_features`` for an input of shape
            (T, N, num_features).
        block_channels (list): A list of integers indicating the number
            of channels per `TDSConv2dBlock`.
        kernel_width (int): The kernel size of the temporal convolutions.
    """

    def __init__(
        self,
        num_features: int,
        block_channels: Sequence[int] = (24, 24, 24, 24),
        kernel_width: int = 32,
    ) -> None:
        super().__init__()
        self.kernel_width = kernel_width
        self.num_blocks = len(block_channels)

        assert len(block_channels) > 0
        tds_conv_blocks = []
        for channels in block_channels:
            feature_width = num_features // channels
            assert (
                num_features % channels == 0
            ), f"block_channels {channels} must evenly divide num_features {num_features}"
            tds_conv_blocks.extend(
                [
                    TDSConv2dBlock(channels, feature_width, kernel_width),
                    TDSFullyConnectedBlock(num_features),
                ]
            )
        self.tds_conv_blocks = nn.Sequential(*tds_conv_blocks)

    def forward(self, inputs: torch.Tensor) -> torch.Tensor:
        return self.tds_conv_blocks(inputs)  # (T, N, num_features)

class TdsStage(nn.Module):
    def __init__(
        self,
        in_channels: int = 16,
        in_conv_kernel_width: int = 5,
        in_conv_stride: int = 1,
        num_blocks: int = 1,
        channels: int = 8,
        feature_width: int = 2,
        kernel_width: int = 1,
        out_channels: int | None = None,
    ):
        super().__init__()
        """Stage of several TdsBlocks preceded by a non-separable sub-sampling conv.

        The initial (and optionally sub-sampling) conv layer maps the number of
        input channels to the corresponding internal width used by the residual TDS
        blocks.

        Follows the multi-stage network construction from
        https://arxiv.org/abs/1904.02619.
        """

        layers: collections.OrderedDict[str, nn.Module] = collections.OrderedDict()

        C = channels * feature_width

        self.out_channels = out_channels

        # Conv1d block
        if in_conv_kernel_width > 0:
            layers["conv1dblock"] = Conv1dBlock(
                in_channels,
                C,
                kernel_size=in_conv_kernel_width,
                stride=in_conv_stride,
            )
        elif in_channels != C:
            # Check that in_channels is consistent with TDS
            # channels and feature width
            raise ValueError(
                f"in_channels ({in_channels}) must equal channels *"
                f" feature_width ({channels} * {feature_width}) if"
                " in_conv_kernel_width is not positive."
            )

        # TDS block
        layers["tds_block"] = TDSConvEncoder(
            num_features=C,
            block_channels=[channels] * num_blocks,
            kernel_width=kernel_width,
        )

        # Linear projection
        if out_channels is not None:
            self.linear_layer = nn.Linear(channels * feature_width, out_channels)

        self.layers = nn.Sequential(layers)

    def forward(self, x):
        x = self.layers(x)
        if self.out_channels is not None:
            x = self.linear_layer(x.swapaxes(-1, -2)).swapaxes(-1, -2)

        return x


class TdsNetwork(nn.Module):
    def __init__(
        self, conv_blocks: Sequence[Conv1dBlock], tds_stages: Sequence[TdsStage]
    ):
        super().__init__()
        self.layers = nn.Sequential(*conv_blocks, *tds_stages)
        self.left_context = self._get_left_context(conv_blocks, tds_stages)
        self.right_context = 0

    def forward(self, x):
        return self.layers(x)

    def _get_left_context(self, conv_blocks, tds_stages) -> int:
        left, stride = 0, 1

        for conv_block in conv_blocks:
            left += (conv_block.kernel_size - 1) * stride
            stride *= conv_block.stride

        for tds_stage in tds_stages:

            conv_block = tds_stage.layers.conv1dblock
            left += (conv_block.kernel_size - 1) * stride
            stride *= conv_block.stride

            tds_block = tds_stage.layers.tds_block
            for _ in range(tds_block.num_blocks):
                left += (tds_block.kernel_width - 1) * stride

        return left


class SequentialLSTM(nn.Module):
    """
    LSTM where each forward() call computes only a single time step, to be compatible
    looping over time manually.

    NOTE: Need to manually reset the state in outer context after each trajectory!
    """

    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        hidden_size: int,
        num_layers: int = 1,
        scale: float = 1.0,
    ):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.lstm = nn.LSTM(in_channels, hidden_size, num_layers, batch_first=True)
        self.hidden: tuple[torch.Tensor, torch.Tensor] | None = None
        self.mlp_out = nn.Sequential(
            nn.LeakyReLU(), nn.Linear(hidden_size, out_channels)
        )
        self.scale = scale

    def reset_state(self):
        self.hidden = None

    def forward(self, x):
        """Forward pass for a single time step, where x is (batch, channel.)"""

        if self.hidden is None:
            # Initialize hidden state with zeros
            batch_size = x.size(0)
            device = x.device
            size = (self.num_layers, batch_size, self.hidden_size)
            self.hidden = (torch.zeros(*size).to(device), torch.zeros(*size).to(device))

        out, self.hidden = self.lstm(x[:, None], self.hidden)
        return self.mlp_out(out[:, 0]) * self.scale

    def _non_sequential_forward(self, x):
        """Non-sequential forward pass, where x is (batch, time, channel)."""
        return self.mlp_out(self.lstm(x)[0]) * self.scale

In [ ]:
import torch

from torch import nn
from torch.nn.functional import interpolate


EMG_SAMPLE_RATE = 2000


class BasePoseModule(nn.Module):
    """
    Pose module consisting of a network with a left and right context. Predictions span
    the inputs[left_context : -right_context], and are upsampled to match the sample
    rate of the inputs.
    """

    def __init__(
        self,
        network: nn.Module,
        out_channels: int = 20,
    ):
        super().__init__()
        self.network = network
        self.out_channels = out_channels

        self.left_context = network.left_context
        self.right_context = network.right_context

    def forward(
        self, batch: dict[str, torch.Tensor], provide_initial_pos: bool
    ) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:

        emg = batch["emg"]
        joint_angles = batch["joint_angles"]
        no_ik_failure = batch["no_ik_failure"]

        # Get initial position
        initial_pos = joint_angles[..., self.left_context]
        if not provide_initial_pos:
            initial_pos = torch.zeros_like(initial_pos)

        # Generate prediction
        pred = self._predict_pose(emg, initial_pos)

        # Slice joint angles to match the span of the predictions
        start = self.left_context
        stop = None if self.right_context == 0 else -self.right_context
        joint_angles = joint_angles[..., slice(start, stop)]
        no_ik_failure = no_ik_failure[..., slice(start, stop)]

        # Match the sample rate of the predictions to that of the joint angles
        n_time = joint_angles.shape[-1]
        pred = self.align_predictions(pred, n_time)
        no_ik_failure = self.align_mask(no_ik_failure, n_time)

        return pred, joint_angles, no_ik_failure

    def _predict_pose(self, emg: torch.Tensor, initial_pos: torch.Tensor):
        raise NotImplementedError

    def align_predictions(self, pred: torch.Tensor, n_time: int):
        """Temporally resamples predictions to match the length of targets."""
        return interpolate(pred, size=n_time, mode="linear")

    def align_mask(self, mask: torch.Tensor, n_time: int):
        """Temporally resample mask to match the length of targets."""
        # 2D Inputs don't work for interpolate(), so we add a dummy channel dimension
        mask = mask[:, None].to(torch.float32)
        aligned = interpolate(mask, size=n_time, mode="nearest")
        return aligned.squeeze(1).to(torch.bool)


class PoseModule(BasePoseModule):
    """
    Tracks pose by predicting posititions or velocities,
    optionally given the initial state.
    """

    def __init__(self, network: nn.Module, predict_vel: bool = False):
        super().__init__(network)
        self.predict_vel = predict_vel

    def _predict_pose(self, emg: torch.Tensor, initial_pos: torch.Tensor):
        pred = self.network(emg)  # BCT
        if self.predict_vel:
            pred = initial_pos[..., None] + torch.cumsum(pred, -1)
        return pred


class StatePoseModule(BasePoseModule):
    """
    Tracks pose by predicting posititions or velocities, optionally given the initial
    state and conditioned on the previous state at each time point.
    """

    def __init__(
        self,
        network: nn.Module,
        decoder: nn.Module,
        state_condition: bool = True,
        predict_vel: bool = False,
        rollout_freq: int = 50,
    ):
        super().__init__(network)
        self.decoder = decoder
        self.state_condition = state_condition
        self.predict_vel = predict_vel
        self.rollout_freq = rollout_freq

    def _predict_pose(self, emg: torch.Tensor, initial_pos: torch.Tensor):

        features = self.network(emg)  # BCT
        preds = [initial_pos]

        # Resample features to rollout frequency
        seconds = (
            emg.shape[-1] - self.left_context - self.right_context
        ) / EMG_SAMPLE_RATE
        n_time = round(seconds * self.rollout_freq)
        features = interpolate(features, n_time, mode="linear", align_corners=True)

        # Reset LSTM hidden state
        if isinstance(self.decoder, SequentialLSTM):
            self.decoder.reset_state()

        for t in range(features.shape[-1]):

            # Prepare decoder inputs
            inputs = features[:, :, t]
            if self.state_condition:
                inputs = torch.concat([inputs, preds[-1]], dim=-1)

            # Predict pose
            pred = self.decoder(inputs)
            if self.predict_vel:
                pred = pred + preds[-1]
            preds.append(pred)

        # Remove first pred, because it is the initial_pos (not a network prediction)
        return torch.stack(preds[1:], dim=-1)

In [ ]:

def resample_emg(emg, orig_hz=500, target_hz=2000):
    """
    emg: numpy array of shape (N, C, T)
    returns: numpy array of shape (N, C, T_new)
    """
    emg = np.asarray(emg, dtype=np.float32)

    N, C, T = emg.shape
    scale = target_hz / orig_hz
    T_new = int(round(T * scale))

    # Original and target time indices
    x_old = np.linspace(0, T - 1, T)
    x_new = np.linspace(0, T - 1, T_new)

    emg_resampled = np.empty((N, C, T_new), dtype=np.float32)

    for n in range(N):
        for c in range(C):
            emg_resampled[n, c] = np.interp(
                x_new,
                x_old,
                emg[n, c]
            )

    return emg_resampled


In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader


class VEMG2PoseNPYDatasetFast(Dataset):
    def __init__(self, subject_ids, allSubjects, x_prefix="X_", y_prefix="Y_"):
        self.subject_ids = subject_ids
        self.allSubjects = allSubjects
        self.x_prefix = x_prefix
        self.y_prefix = y_prefix

        # Build global index (subject, sample_idx)
        self.index = []
        self.lengths = {}
        for sid in subject_ids:
            x = np.load(f"{x_prefix}{allSubjects[sid]}.npy", mmap_mode="r")
            self.lengths[sid] = len(x)
            self.index.extend((sid, i) for i in range(len(x)))

        # Per-worker cache (file handles only, not full arrays)
        self._x_mm = {}
        self._y_mm = {}

    def _get_memmaps(self, sid):
        if sid not in self._x_mm:
            self._x_mm[sid] = np.load(f"{self.x_prefix}{self.allSubjects[sid]}.npy", mmap_mode="r")
            self._y_mm[sid] = np.load(f"{self.y_prefix}{self.allSubjects[sid]}.npy", mmap_mode="r")
        return self._x_mm[sid], self._y_mm[sid]

    def __len__(self):
        return len(self.index)

    def __getitem__(self, i):
        sid, sample_idx = self.index[i]
        x_mm, y_mm = self._get_memmaps(sid)

        x = np.array(x_mm[sample_idx], copy=True)
        y = np.array(y_mm[sample_idx], copy=True)

        return torch.from_numpy(x), torch.from_numpy(y)


def train_vemg2pose(
    trainIdx,
    num_epochs=5000,
    lr=1e-3,
    batch_size=256,
    device=None,
):
    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"

    B, _, T = 256, 8, 6000
    num_joints = 15

    # =====================
    # Train / Val split
    # =====================


    # Select one validation subject randomly
    valIdx = int(np.random.choice(trainIdx, 1)[0])
    trainIdx = trainIdx[trainIdx != valIdx]

    # =====================
    # DataLoaders
    # =====================
    train_loader = DataLoader(
        VEMG2PoseNPYDatasetFast(trainIdx, allSubjects),
        batch_size=batch_size,
        shuffle=True,
        pin_memory=True,
        num_workers=4,
    )

    val_loader = DataLoader(
        VEMG2PoseNPYDatasetFast([valIdx], allSubjects),
        batch_size=batch_size,
        shuffle=False,
        pin_memory=True,
        num_workers=4
    )

    # =====================
    # Build the model
    # =====================

    conv_blocks = [
        Conv1dBlock(N_EMG_CHANNEL, 256, kernel_size=11, stride=5),
        Conv1dBlock(256, 256, kernel_size=5, stride=2),
    ]

    tds_stages = [
        TdsStage(
            in_channels=256,
            in_conv_kernel_width=17,
            in_conv_stride=4,
            num_blocks=2,
            channels=16,
            feature_width=16,
            kernel_width=9,
        ),
        TdsStage(
            in_channels=256,
            in_conv_kernel_width=9,
            in_conv_stride=2,
            num_blocks=2,
            channels=16,
            feature_width=16,
            kernel_width=5,
            out_channels=64,
        ),
    ]

    encoder = TdsNetwork(conv_blocks, tds_stages)

    decoder = SequentialLSTM(
        in_channels=64 + num_joints,
        out_channels=num_joints,
        hidden_size=256,
        num_layers=2,
    )

    model = StatePoseModule(
        network=encoder,
        decoder=decoder,
        state_condition=True,
        predict_vel=True,
        rollout_freq=50,
    ).to(device)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=lr,
        weight_decay=1e-4,
    )

    # =====================
    # Training loop
    # =====================

    lossEvol = []
    minValLoss = float("inf")
    minValEpoch = 0

    for epoch in range(num_epochs):

        # -------- TRAIN --------
        model.train()
        train_loss = 0.0

        for x_cpu, y_cpu in train_loader:

            x = x_cpu.to(device, non_blocking=True)
            y = y_cpu.to(device, non_blocking=True)
            bs = x.shape[0]

            batch = {
                "emg": x,
                "joint_angles": y,
                "no_ik_failure": torch.ones(
                    bs, T, dtype=torch.bool, device=device
                ),
            }

            pred, gt, mask = model(batch, provide_initial_pos=True)

            loss = (pred - gt) ** 2
            loss = loss * mask[:, None, :]
            loss = loss.sum() / mask.sum()

            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            train_loss += loss.item()

            del x, y, batch, pred, gt, mask, loss

        train_loss /= len(train_loader)

        # -------- VALIDATION --------
        model.eval()
        val_loss = 0.0

        with torch.no_grad():
            for x_cpu, y_cpu in val_loader:
                x = x_cpu.to(device, non_blocking=True)
                y = y_cpu.to(device, non_blocking=True)
                bs = x.shape[0]

                batch = {
                    "emg": x,
                    "joint_angles": y,
                    "no_ik_failure": torch.ones(
                        bs, T, dtype=torch.bool, device=device
                    ),
                }

                pred, gt, mask = model(batch, provide_initial_pos=True)

                loss = (pred - gt) ** 2
                loss = loss * mask[:, None, :]
                loss = loss.sum() / mask.sum()

                val_loss += loss.item()

                del x, y, batch, pred, gt, mask, loss

        val_loss /= len(val_loader)

        print(
            f"Epoch {epoch:04d} | "
            f"train loss {train_loss:.6f} | "
            f"val loss {val_loss:.6f}"
        )

        lossEvol.append((train_loss, val_loss))
        if val_loss < minValLoss:
            minValLoss = val_loss
            minValEpoch = epoch
        elif epoch - minValEpoch > 10:
            print(f"Early stopping at epoch {epoch}")
            break

    gc.collect()
    torch.cuda.empty_cache()

    return model, np.array(lossEvol)


In [ ]:
def predict_vemg2pose(
    model,
    x,
    num_joints,
    initial_pose=None,
    batch_size=256,
):
    """
    model: trained vemg2pose (StatePoseModule)
    x: (B, 8, T) EMG
    num_joints: int
    initial_pose: (B, num_joints) or None
    """
    device = next(model.parameters()).device
    model.eval()

    B, _, T = x.shape
    x = torch.from_numpy(x).to(device)

    # allocate output tensor
    preds = torch.zeros(
        B, num_joints,
        device=device,
        dtype=x.dtype,
    )

    with torch.no_grad():
        for i in range(0, B, batch_size):
            xb = x[i : i + batch_size]
            bs = xb.shape[0]

            if initial_pose is None:
                joint_angles = torch.zeros(
                    bs, num_joints, T, device=device
                )
                provide_initial_pos = False
            else:
                joint_angles = (
                    initial_pose[i : i + batch_size]
                    .to(device)[..., None]
                    .repeat(1, 1, T)
                )
                provide_initial_pos = True

            batch = {
                "emg": xb,
                "joint_angles": joint_angles,
                "no_ik_failure": torch.ones(
                    bs, T, dtype=torch.bool, device=device
                ),
            }

            pred, _, _ = model(
                batch,
                provide_initial_pos=provide_initial_pos,
            )

            preds[i : i + bs] = pred[:, :, -1]

            # free per-batch tensors early
            del xb, joint_angles, batch, pred

    del x
    torch.cuda.empty_cache()

    return preds.cpu().numpy()


In [ ]:
def vemg2pose(trainIdx, testIdx):
    model, loss = train_vemg2pose(trainIdx, num_epochs=50, batch_size=64, lr=1e-3) # batch size of 64, lr of 1e-3 from vemg2pose paper
    plt.rcParams['figure.figsize'] = [5, 5]
    plt.plot(loss[:, 0])
    plt.plot(loss[:, 1])
    plt.show()


    x = np.load(f"X_{allSubjects[testIdx]}.npy")
    yPred = predict_vemg2pose(model, x, num_joints=N_KINEMATIC_CHANNEL).astype(np.float32)
    del model

    return yPred

# Model evaluation

In [ ]:
def getsync_data(subjectName, labelShift, windowSize, multipleFreqBands=False, labelSequence=False):
  emg, label = loadData(subjectName)
  label_sync = sync(emg, label)

  if multipleFreqBands:
    x = []
    sfreq = EMG_S_FREQ
    ch_names = [f"emg{i+1}" for i in range(N_EMG_CHANNEL)]
    info = mne.create_info(ch_names=ch_names, sfreq=sfreq, ch_types="emg")
    bands = [(15, 40), (40, 80), (80, 150)]
    for lb, hb in bands:
      emg_band = mne.io.RawArray(deepcopy(emg.T), info)
      emg_band.filter(lb, hb, picks=ch_names)
      emg_band = emg_band.get_data().T
      x_band, y = extractSamples_10fps(emg_band, label_sync, shift=labelShift, size=windowSize)
      x.append(x_band)
    x = np.array(x).transpose((1, 0, 2, 3))
    return x, y

  else:
    x, y = extractSamples_10fps(emg, label_sync, shift=labelShift, size=windowSize, labelSequence=labelSequence)
    return x, y

In [ ]:
def cross_val_prediction_loso(x, y, featureExtractor, model, sequenceLength):

  for subject in range(len(x)): # emg standardization per subject
    for c in range(8):
        if len(x[subject].shape) == 3: # standardization
          m = np.mean(x[subject][:, c, :])
          s = np.std(x[subject][:, c, :])
          x[subject][:, c, :] = (x[subject][:, c, :] - m) / s
        elif len(x[subject].shape) == 4:
          for band in range(x[subject].shape[1]): # standardization when we have multiple frequency bands of signal
            m = np.mean(x[subject][:, band, c, :])
            s = np.std(x[subject][:, band, c, :])
            x[subject][:, band, c, :] = (x[subject][:, band, c, :] - m) / s

  for foldId in range(len(x)):
    print(f"-- Fold {foldId+1}/{len(x)} --")

    trainIdx = np.arange(len(x))
    testIdx = foldId
    trainIdx = np.delete(trainIdx, testIdx)

    xTrain, xTest = featureExtractor(np.concatenate(deepcopy(x[trainIdx])), x[testIdx])
    yTrain, yTest = deepcopy(np.concatenate(y[trainIdx])), deepcopy(y[testIdx])

    xTrain, yTrain = createSequences(xTrain, yTrain, sequenceLength=sequenceLength)
    xTest, yTest = createSequences(xTest, yTest, sequenceLength=sequenceLength)

    labelScaler = StandardScaler().fit(yTrain)
    yTrain_scaled = labelScaler.transform(yTrain)

    yPred = model(xTrain, yTrain_scaled, xTest)

    yPred_unscaled = labelScaler.inverse_transform(yPred)

    groundTruth = np.array(yTest)
    yPred_unscaled = np.array(yPred_unscaled)

    nse = evaluate(yPred_unscaled, groundTruth)

In [ ]:
import pickle

def cross_val_prediction_loso_forCNN(featureExtractor, model, sequenceLength, labelEncoder=None):
  try:
    with open("label_scaler.pkl", "rb") as f:
      labelScaler = pickle.load(f)
  except:
    for subject in range(len(allSubjects)):
      print(f"standardize X subject {subject}")
      x = np.load(f"X_{allSubjects[subject]}.npy")
      for c in range(8):
        if len(x.shape) == 3: # standardization
          m = np.mean(x[:, c, :])
          s = np.std(x[:, c, :])
          x[:, c, :] = (x[:, c, :] - m) / s
      x, _ = featureExtractor(x, x[:2]) # will only be raw or envelope
      np.save(f"X_{allSubjects[subject]}.npy", x)
      del x

    y = np.load(f"Y_{allSubjects[0]}.npy")
    if len(y.shape) == 2:
      labelScaler = StandardScaler().fit(y)
    elif len(y.shape) == 3:
      labelScaler = StandardScaler().fit(y[:, :, -1])
    del y

    with open("label_scaler.pkl", "wb") as f:
      pickle.dump(labelScaler, f)

    for subject in range(len(allSubjects)):
      print(f"standardize Y subject {subject}")
      y = np.load(f"Y_{allSubjects[subject]}.npy")
      if len(y.shape) == 2:
        y = labelScaler.transform(y)
      elif len(y.shape) == 3:
        for i in range(y.shape[2]):
          y[:, :, i] = labelScaler.transform(y[:, :, i])
      np.save(f"Y_{allSubjects[subject]}.npy", y)
      del y

  for foldId in range(len(allSubjects)):
    print(f"-- Fold {foldId+1}/{len(allSubjects)} --")

    trainIdx = np.arange(len(allSubjects))
    testIdx = foldId
    trainIdx = np.delete(trainIdx, testIdx)

    yPred = model(trainIdx, testIdx)

    yPred_unscaled = labelScaler.inverse_transform(yPred)

    groundTruth = np.load(f"Y_{allSubjects[testIdx]}.npy").astype(np.float32)
    if len(groundTruth.shape) == 3:
      groundTruth = groundTruth[:, :, -1]
    groundTruth = labelScaler.inverse_transform(groundTruth)
    nse = evaluate(yPred_unscaled, groundTruth)

In [ ]:
def evaluate(prediction, groundTruth):
  nse = np.array([(prediction[:, i] - groundTruth[:, i])**2 /np.var(groundTruth[:, i]) for i in range(15)])
  ae = np.array([(np.abs(prediction[:, i] - groundTruth[:, i])) for i in range(15)])

  nse2 = deepcopy(nse)
  nse3 = deepcopy(nse)
  for i in range(15):
    nse2[i][nse2[i] > np.percentile(nse2[i], 95)] = np.nan
    nse3[i][nse3[i] > np.percentile(nse3[i], 90)] = np.nan

  print(f"Average NSE : {round(np.mean(nse), 4)} +/- {round(np.std(nse), 4)}")
  print(f"Median NSE : {round(np.median(nse), 4)}")
  print(f"trimmed mean (5%-trim) NSE : {round(np.nanmean(nse2), 4)} +/- {round(np.nanstd(nse2), 4)}")
  print(f"trimmed mean (10%-trim) NSE : {round(np.nanmean(nse3), 4)} +/- {round(np.nanstd(nse3), 4)}")
  print(f"Thresholded NSE Rate (0.4) : {round(np.mean(100 * (nse < 0.4)), 2)}%")
  print(f"Thresholded NSE Rate (0.3) : {round(np.mean(100 * (nse < 0.3)), 2)}%")
  print(f"Thresholded NSE Rate (0.2) : {round(np.mean(100 * (nse < 0.2)), 2)}%")
  print(f"Thresholded NSE Rate (0.1) : {round(np.mean(100 * (nse < 0.1)), 2)}%")

  plt.rcParams['figure.figsize'] = [10, 3]
  fig, ax = plt.subplots(1, 2)
  sn.boxplot(nse.T, showfliers=False, ax=ax[0])
  ax[0].grid()
  ax[0].set_title("NMSE")
  ax[0].set_xlabel("Joint ID")
  ax[0].set_ylabel("NMSE")

  sn.boxplot(ae.T, showfliers=False, ax=ax[1])
  ax[1].grid()
  ax[1].set_title("Absolute Error")
  ax[1].set_xlabel("Joint ID")
  ax[1].set_ylabel("Absolute Error (degrees)")
  plt.show()

  return nse

In [ ]:
def plot_results(prediction, groundTruth):
  plt.rcParams['figure.figsize'] = [20, 7]
  maxIdx = 2000
  plt.plot(groundTruth[:maxIdx] + np.arange(15)*200, color="black")
  plt.plot(prediction[:maxIdx] + np.arange(15)*200, color="tab:blue")
  for i in range(15):
    plt.fill_between(np.arange(maxIdx), groundTruth[:maxIdx, i] + i*200, prediction[:maxIdx, i] + i*200, color="tab:red", alpha=0.5)
  plt.show()

  plt.rcParams['figure.figsize'] = [20, 5]
  plt.title("Distributions of real angles (blue) and predicted angles (orange)")
  plt.violinplot(groundTruth, side="low")
  plt.violinplot(prediction, side="high")
  plt.show()

  plt.rcParams['figure.figsize'] = [50, 3]
  fig, ax = plt.subplots(1, 15)
  for angle in range(15):
    ax[angle].scatter(groundTruth[:, angle], prediction[:, angle], s=1, alpha=0.1)
    ax[angle].plot([min(groundTruth[:, angle]), max(groundTruth[:, angle])], [min(groundTruth[:, angle]), max(groundTruth[:, angle])], color="tab:red", linestyle="--")
    ax[angle].set_title(f"Angle {angle+1}")
    ax[angle].set_xlabel("Ground truth")
    ax[angle].set_ylabel("Prediction")
  plt.show()

# Experiment : our Method (TRR)

In [ ]:
model = rnn
featureExtractor = covMatTangentSpace_3FreqBands
windowSize = 150
sequenceLength = 10
labelShift = 0

In [ ]:
X = []
Y = []
for subjectName in allSubjects:
  print(f"------------ loading data from {subjectName} -----------------")
  x, y = getsync_data(subjectName, labelShift, windowSize, multipleFreqBands=True)
  X.append(x)
  Y.append(y)
X = np.array(X, dtype=object)
Y = np.array(Y, dtype=object)

In [ ]:
cross_val_prediction_loso(X, Y, featureExtractor, model, sequenceLength)

 # Experiment : MLP on CMTS

In [ ]:
model = mlp
featureExtractor = covMatTangentSpace
windowSize = 300
sequenceLength = 1
labelShift = 0

In [ ]:
X = []
Y = []
for subjectName in allSubjects:
  print(f"------------ loading data from {subjectName} -----------------")
  x, y = getsync_data(subjectName, labelShift, windowSize)
  X.append(x)
  Y.append(y)
X = np.array(X, dtype=object)
Y = np.array(Y, dtype=object)

In [ ]:
cross_val_prediction_loso(X, Y, featureExtractor, model, sequenceLength)

 # MLP on TDF

In [ ]:
model = mlp
featureExtractor = timeDomainFeatures
windowSize = 300
sequenceLength = 1
labelShift = 0

In [ ]:
X = []
Y = []
for subjectName in allSubjects:
  print(f"------------ loading data from {subjectName} -----------------")
  x, y = getsync_data(subjectName, labelShift, windowSize)
  X.append(x)
  Y.append(y)
X = np.array(X, dtype=object)
Y = np.array(Y, dtype=object)

In [ ]:
cross_val_prediction_loso(X, Y, featureExtractor, model, sequenceLength)

# CRNN

In [ ]:
model = deep_crnn
featureExtractor = rawEMG # switch to rawnEMG or envelopeEMG
windowSize = 600
sequenceLength = 1
labelShift = 0

In [ ]:
X = []
Y = []
for subjectName in allSubjects:
  print(f"------------ loading data from {subjectName} -----------------")
  x, y = getsync_data(subjectName, labelShift, windowSize)
  np.save(f"X_{subjectName}.npy", x)
  np.save(f"Y_{subjectName}.npy", y)

In [ ]:
cross_val_prediction_loso_forCNN(featureExtractor, model, sequenceLength)

# vemg2pose

In [ ]:
model = vemg2pose
featureExtractor = rawEMG
windowSize = 1500
sequenceLength = 1
labelShift = 0

In [ ]:
X = []
Y = []
for subjectName in allSubjects:
  print(f"------------ loading data from {subjectName} -----------------")
  x, y = getsync_data(subjectName, labelShift, windowSize, labelSequence=True)
  np.save(f"X_{subjectName}.npy", resample_emg(x))
  np.save(f"Y_{subjectName}.npy", resample_emg(y).astype(np.float16))

In [ ]:
cross_val_prediction_loso_forCNN(featureExtractor, model, sequenceLength)